# M2 CLV-conditioned Modulation LightGCN
Dunnhumby seed 42 validation에서 순수 LightGCN(M1)과 M2 한 모형만 비교합니다. N/V 행동표현은 별도 추천점수를 만들지 않고, 기존 64차원 ID 임베딩을 차원별로 최대 ±10% 조절합니다. Test와 holdout은 만들거나 평가하지 않습니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess

drive.mount('/content/drive')
REVIEWED_SHA = '1b1964e63b2b5229c502d5ad2ba1331efc816f53'
REPO = Path('/content/clv-m2-lightgcn-runner')
os.chdir('/content')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REVIEWED_SHA], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert head == REVIEWED_SHA, (head, REVIEWED_SHA)
os.chdir(REPO)
print('reviewed source:', head)

## 실행 설정 확인
아래 출력에서 dataset=dunnhumby, seed=42, models=[m1, m2_clv_modulation], eval_test=false, eval_holdout=false인지 확인합니다.

In [ ]:
import json, torch
from lightgcn_clv_modulation import (
    configure_modulation_dunnhumby_run, preflight_summary, run_experiment,
)

assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
cfg = configure_modulation_dunnhumby_run(
    eval_test=False,
    eval_holdout=False,
)
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

## M1과 M2 실행
이 셀 하나가 실제 학습 셀입니다. epoch별 진행상태는 Drive에 저장되어 연결이 끊겨도 같은 설정으로 다시 실행하면 자동 재개됩니다.

In [ ]:
result_df = run_experiment(cfg)

## Validation 결과

In [ ]:
from IPython.display import display
import pandas as pd

columns = [
    'model_id', 'role', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'revenue@10', 'revenue@20', 'revenue@50',
    'arp@10', 'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
    'value_alignment', 'tau', 'user_n_abs_mean', 'user_v_abs_mean',
    'item_n_abs_mean', 'item_v_abs_mean', 'combined_saturation_share',
]
available = [name for name in columns if name in result_df.columns]
display(result_df[result_df['split'].eq('val')][available].sort_values(['role', 'model_id']))
print('screening 판정:', result_df.attrs['screening_decision'])
print('사용자 N/V 유효 마스크:', result_df.attrs['user_axis_mask_summary'])
print('
M1 대비 paired delta:')
display(pd.read_csv(result_df.attrs['result_paths']['delta_csv']))
print('결과 파일:', result_df.attrs['result_paths'])